# Resource Estimation for fault-tolerant implementation of QPE

We can now estimate the resources required for a fault-tolerant implementation of QPE for the deduced Hamiltonian. For this, we have to make some assumptions first.
The paper uses a *rotated surface code* for error correction with a certain *code distance* and implements certain gates using lattice surgery and magic states.
They assume a physical error rate of $p = 10^{-3}$ or $p = 10^{-4}$. This refers to the probability of a physical qubit experiencing an error during an operation. Further, they assume a target failure probability of 1% for the entire computation. This means that the probability of the entire quantum computation failing should be less than 1%.
Now, they assume a certain noise model from that we can deduce the probability of a logical error (i.e., an error that occurs to our encoded logical qubits) as a function of the code distance $d$ and the physical error rate $p$. The paper uses the following formula for the logical error rate:
$$
p_L(d, p) = 0.1 \left(100p \right)^{(d+1)/2}
$$

The probability for a logical error to occur during the entire computation is given by:
$$
P_L(d, p) = (n_{data} + n_{route}) \cdot n_{meas} \cdot p_L(d, p)
$$
Here, $n_{data}$ is the number of logical qubits used for data, $n_{route}$ is the number of ancillas we need for lattice surgery and $n_{meas}$ is the number of QEC rounds.
Now, we can solve this for the code distance $d$ given a target failure probability $P_L(d, p) < 0.005$ and a physical error rate $p$. We can then use this code distance to estimate the number of physical qubits required for the computation.

In our case $n_{data}$ is 2 for the iterative QPE approach, $n_{route}$ is also 2 and $n_{meas}$ is a little more complicated to calculate. For that we need to know the total number of gates we have for each gate type and multiply this by the number of QEC rounds needed for each gate type.
These are:
| Logical Gate / Operation | QEC Rounds Formula |
| :--- | :--- |
| **CNOT** | $3d + 4$ |
| **Hadamard ($H$)** | $3d + 4$ |
| **$T$ / $T^\dagger$-like** | $\frac{5d}{2} + 4$ |
| **$S$ / $S^\dagger$** | $\frac{3d}{2} + 3$ |
| **$Z$-basis Measurement** | $1$ |

In [ ]:
import ipywidgets as widgets
from ipywidgets import interact
import matplotlib.pyplot as plt
import numpy as np

# Fixed Circuit Parameters
n_data = 2
n_route = 2
target_error = 0.005


gate_cost = {
    'CX': (3, 4),
    'H': (3, 4),
    'T': (2.5, 4),
    'S': (1.5, 3),
    'M': (0, 1),
}

# TODO: Miranda pls implement -><-, M is for measurement
gate_numbers = {'CX': 34, 'H': 411, 'S': 12, 'T': 386, 'M': 3}


def n_meas(distance):
    return sum([
        gate_count * (distance * gate_cost[gate][0] + gate_cost[gate][1])
        for gate, count in gate_numbers.items()
        for gate, gate_count in [(gate, count)]
    ])


def prob_logical_error(distance, p_e):
    p_L = 0.1 * ((100.0 * p_e) ** ((distance + 1.0) / 2.0))
    return (n_data + n_route) * n_meas(distance) * p_L


@interact(
    p_e=widgets.FloatLogSlider(
        value=1e-3,
        min=-4,
        max=-2.7,
        step=0.05,
        description='p_e:',
        readout_format='.1e',
    )
)
def plot_logical_error(p_e):
    d_min, d_max = 1, 20
    distances = np.arange(d_min, d_max + 1, 2)
    errors = [prob_logical_error(d, p_e) for d in distances]

    plt.figure(figsize=(8, 5))

    # Logical Error Curve
    plt.plot(
        distances,
        errors,
        'o-',
        color='navy',
        linewidth=2,
        label=f'Logical Error ($p_e = {p_e:.1e}$)',
    )

    # Target Budget Line (0.005)
    plt.axhline(
        y=target_error,
        color='crimson',
        linestyle='--',
        linewidth=2,
        label=f'Target Budget ({target_error})',
    )

    plt.yscale('log')
    plt.xticks(distances)
    plt.xlabel('Code Distance ($d$)', fontsize=11)
    plt.ylabel('Total Logical Error Probability', fontsize=11)
    plt.title('Logical Error Probability vs. Code Distance', fontsize=12)
    plt.grid(True, which='both', linestyle='--', alpha=0.5)
    plt.legend(fontsize=10)

    plt.tight_layout()
    plt.show()

interactive(children=(FloatLogSlider(value=0.001, description='p_e:', max=-2.7, min=-4.0, readout_format='.1e'…